In [1]:
from class_AEdriver_rotation import ExtendedAEdriver

In [2]:
AEdriver = ExtendedAEdriver(COM='COM5', baudrate=38400)

In [3]:
AEdriver.initialize()

In [13]:

import binascii
import serial
import time
from typing import Union
def rx_to_int32(
    rx: Union[bytes, str],
    *,
    verify_checksum: bool = True,
) -> int:
    """
    Extract the DATA field of an 8-bit-length framed packet, reverse its endianness
    and return it as a 32-bit *unsigned* integer.
    Packet format
    -------------
        [0] LEN        : total bytes, **including** this LEN byte
        [1] ADDR       : address
        [2] CMD        : command / status
        [3:-1] DATA    : N-byte payload   (N = LEN − 4)
        [-1] CHKSUM    : 8-bit sum of LEN+ADDR+CMD+DATA  (mod 256)
    Parameters
    ----------
    rx : ``bytes`` or *hex-encoded* ``str``
        Raw packet as received.
    verify_checksum : bool, optional
        If True (default) raise ``ValueError`` when the checksum is wrong.
    Returns
    -------
    int
        Little-endian value of the DATA field (0 – 4 294 967 295).
    Example
    -------
    >>> rx_to_int32("080104700000007D")
    112
    """
    # ── accept either bytes or hex string ───────────────────────────────────
    if isinstance(rx, str):
        try:
            rx_bytes = bytes.fromhex(rx)
        except ValueError as e:
            raise ValueError("RX hex string is malformed") from e
    else:
        rx_bytes = bytes(rx)
    if len(rx_bytes) < 5:
        raise ValueError("Packet too short")
    # ── length check ───────────────────────────────────────────────────────
    pkt_len = rx_bytes[0]
    if pkt_len != len(rx_bytes):
        raise ValueError(f"Length mismatch: LEN={pkt_len}, actual={len(rx_bytes)}")
    # ── checksum check (optional) ──────────────────────────────────────────
    if verify_checksum:
        calc = sum(rx_bytes[:-1]) & 0xFF
        if calc != rx_bytes[-1]:
            raise ValueError(
                f"Checksum mismatch: expected 0x{rx_bytes[-1]:02X}, calculated 0x{calc:02X}"
            )
    # ── extract DATA bytes ─────────────────────────────────────────────────
    data_bytes = rx_bytes[3:-1]              # may be 0–252 B
    if not data_bytes:
        raise ValueError("No DATA field present")
    # ── reverse endianness and convert to int ──────────────────────────────
    value = int.from_bytes(data_bytes[::-1], byteorder="big", signed=False)
    return value

In [12]:
response = AEdriver.read_position()
print(rx_to_int32(binascii.hexlify(response).decode().upper()))

position: -55894289


TypeError: a bytes-like object is required, not 'NoneType'

In [13]:
EAEdriver.send_cmd('05008022') # highrate
EAEdriver.send_cmd('05008023') # acc

b''